# Statements

The ClickHouse SQL dialect have special statements, as well as recular SQL satements have different meaning or implementation. This pace considers this features of the ClickHouse dialect.

## Array Join

The `ARRAY JOIN` clause is a unique to the ClickHouse dialect. It creates a column containing all the unique elements of the array from column of arrays, while all the other elements of the row are duplicated.

Check more about this clause in the [ARRAY JOIN Clause](https://clickhouse.com/docs/sql-reference/statements/select/array-join) page of the reference.

Two types of `ARRAY JOIN` are supported:

- `ARRAY JOIN`: records with empty arrays are excluded from the result. This can be interpreted as an `INNER` join, meaning that items with empty arrays are not matched with values.
- `LEFT ARRAY JOIN`: garantees that all items of the original table (LEFT) are included to the output.

---

The following cell creates the table containing the array column that we'll use as an example.

In [65]:
--ClickHouse
DROP TABLE IF EXISTS arrays;

CREATE TABLE arrays
(
    idx UInt8,
    arr Array(UInt8)
)
ENGINE = Memory;

INSERT INTO arrays VALUES
    (1, [1, 2]),
    (2, [3, 4]),
    (3, []);

SELECT * FROM arrays;

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,2347791,1051935,3c50c79c-db37-4b4b-86ea-a56ce4e23e34


idx,arr
1,"[1, 2]"
2,"[3, 4]"
3,[]


The output of the `ARRAY JOIN`:

In [66]:
--ClickHouse
SELECT idx, arr, joined
FROM arrays
ARRAY JOIN arr AS joined;

idx,arr,joined
1,"[1, 2]",1
1,"[1, 2]",2
2,"[3, 4]",3
2,"[3, 4]",4


**Note** that the output loses one row that corresnding to the empty array.

If the column with empty array needed to be included to the output use `LEFT ARRAY JOIN`:

In [67]:
--ClickHouse
SELECT idx, arr, joined
FROM arrays
LEFT ARRAY JOIN arr AS joined;

idx,arr,joined
1,"[1, 2]",1
1,"[1, 2]",2
2,"[3, 4]",3
2,"[3, 4]",4
3,[],0


The `joined` value corresponding to the empty array casted to default value for `UInt8`.